# Step 2 — Slide-Level Stratified Split

Split slides into **test** and **train-val** at a 1:7 ratio, stratified by benign/malignant.
The test set (~1,000 images) is held out entirely and never used during training or validation.

### Load catalogs from step 1

Read the per-image and per-slide CSVs produced by step 1. All split logic operates on slides, then maps back to images.

In [2]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data")

slides = pd.read_csv(DATA_DIR / "slide_summary.csv")
images = pd.read_csv(DATA_DIR / "image_catalog.csv")

print(f"{len(slides)} slides, {len(images)} images")

81 slides, 7909 images


### Stratified slide split (1:7 → test : train-val)

Use `train_test_split` on the slides dataframe with `test_size=1/8` (~12.5%), stratified by label.
This keeps the benign/malignant ratio roughly equal in both partitions.

Fixed `random_state` for reproducibility.

In [3]:
SEED = 42
TEST_RATIO = 1 / 8

train_val_slides, test_slides = train_test_split(
    slides,
    test_size=TEST_RATIO,
    stratify=slides["label"],
    random_state=SEED,
)

print(f"Train-val: {len(train_val_slides)} slides")
print(f"Test:      {len(test_slides)} slides")
print(f"\nTrain-val class distribution:")
print(train_val_slides["label"].value_counts())
print(f"\nTest class distribution:")
print(test_slides["label"].value_counts())

Train-val: 70 slides
Test:      11 slides

Train-val class distribution:
label
malignant    49
benign       21
Name: count, dtype: int64

Test class distribution:
label
malignant    8
benign       3
Name: count, dtype: int64


### Map split back to images

Tag every image with its split assignment (`train_val` or `test`) based on which partition its slide landed in. Then verify image counts match expectations (~1,000 test, ~7,000 train-val).

In [4]:
test_slide_ids = set(test_slides["slide_id"])
images["split"] = images["slide_id"].apply(
    lambda s: "test" if s in test_slide_ids else "train_val"
)

print("Images per split:")
print(images["split"].value_counts())
print(f"\nImages per split × class:")
print(images.groupby(["split", "label"]).size().unstack(fill_value=0))
print(f"\nImages per split × magnification:")
print(images.groupby(["split", "magnification"]).size().unstack(fill_value=0))

Images per split:
split
train_val    6801
test         1108
Name: count, dtype: int64

Images per split × class:
label      benign  malignant
split                       
test          282        826
train_val    2198       4603

Images per split × magnification:
magnification   40    100   200   400
split                                
test            267   296   276   269
train_val      1728  1785  1737  1551


### Verify no slide leakage

Critical check — confirm zero overlap between slide IDs in test and train-val. Any overlap here would invalidate the entire evaluation.

In [5]:
train_val_ids = set(train_val_slides["slide_id"])
overlap = test_slide_ids & train_val_ids
assert len(overlap) == 0, f"LEAKAGE: {overlap}"
print(f"No leakage — 0 slides shared between test and train-val.")

No leakage — 0 slides shared between test and train-val.


### Preview test slides

Show which slides ended up in the held-out test set, with their subtype and image count.

In [6]:
test_slides[["slide_id", "label", "subtype", "n_images"]].sort_values("label")

,slide_id,label,subtype,n_images
56,14-22704,benign,PT,158
63,14-29315EF,benign,PT,60
44,14-19854C,benign,TA,64
30,14-16196,malignant,LC,74
25,14-15704,malignant,PC,127
62,14-2773,malignant,DC,144
48,14-20636,malignant,DC,111
12,14-13418DE,malignant,MC,55
33,14-16456,malignant,MC,178
32,14-16448,malignant,DC,53


### Export split assignments

Save the updated image catalog with the `split` column. This is the single source of truth for which images belong to test vs train-val — all downstream notebooks read from this file.

In [7]:
images.to_csv(DATA_DIR / "image_catalog.csv", index=False)
print(f"Updated {DATA_DIR / 'image_catalog.csv'} with split column.")

Updated data/image_catalog.csv with split column.
